In [3]:
from pathlib import Path

import pandas as pd

In [148]:
s1e1_file = Path("/workspaces/tv-time-cs5124/data/transcripts/s1-transcripts/s1_e05_transcript.csv")
data = pd.read_csv(s1e1_file)
data["talking_to"] = ""
data.head(20)

,timestamp,speaker,text,talking_to
0,00:00:50,Mark,Helly leave?,
1,00:00:51,Dylan,Yeah.,
2,00:00:58,Mark,Maybe you should get out of here.,
3,00:01:01,Dylan,"I know. Just wrapping up. Sorry, I love the work.",
4,00:01:29,Mark,Jesus. Help! Help!,
5,00:01:41,Graner,Goddamn it!,
6,00:01:46,Mark,"Oh… Oh, my God.",
7,00:01:59,Mark,Is she okay?,
8,00:02:03,Mark,Helly?,
9,00:02:06,Graner,"Mark, get in the elevator.\nMark: Helly? Helly!",


In [182]:
def make_talk_to(data: pd.DataFrame) -> list[str]:
    """Generates a list of approximately who each speaker is talking to.
    
    Args:
        data: A DataFrame with a "speaker" column.

    Returns:
        List of length `len(data)` where entry i is who `data.loc[i].speaker` was likely talking to.
    """
    i = 0
    talking_to = [""] * len(data)
    current_partipants = set()
    while i < len(data):
        row = data.iloc[i]

        if row.speaker not in current_partipants:
            # We switched to a new conversation
            current_partipants.clear()
            current_partipants.add(row.speaker)

            j = i + 1
            if j == len(data):
                talking_to[i] = talking_to[i-1]
                continue
            while (j < (len(data) - 1)) and (data.iloc[j].speaker in current_partipants):
                # Starting at the current speaker, iterate until you find the other participant
                j += 1
            current_partipants.add(data.iloc[j].speaker)

            continue

        # Get the other person in the conversation
        try:
            other_person = next(p for p in current_partipants if row.speaker != p)
        except StopIteration:
            other_person = "Unknown"
        talking_to[i] = other_person
    
        i += 1
    
    return talking_to

In [188]:
root = Path("/workspaces/tv-time-cs5124/data/transcripts")
new_root = Path("/workspaces/tv-time-cs5124/data/transcripts_with_talking_to")
sub_dirs = {"s1-transcripts", "s2-transcripts"}
for p in root.rglob("*.csv"):
    if p.parent.name not in sub_dirs:
        continue
    data = pd.read_csv(p)
    talking_to = make_talk_to(data)
    data["talking_to"] = talking_to
    new_p = new_root / p.relative_to(root)
    new_p.parent.mkdir(parents=True, exist_ok=True)
    data.to_csv(new_p)